# Under-table pickup: object placement and wrist attachment

Agent handoff · VICAR · 12 October 2026

This notebook documents the implemented change and reproduces it for **all 250 under-table pickup variants**. It also runs with the single-variant standalone RL package. Run all cells with Python, NumPy and SciPy; no GPU, optimizer, Viser server or source TT_PLayer checkout is required. The notebook reads data and checks it; it does not overwrite motions or publish the website.

## Requested behavior

- Object full dimensions **X × Y × Z = 0.04 × 0.04 × 0.25 m**; origin at its geometric center.
- At initial placement: preserve the original object X, subtract **0.04 m in world Y**, and apply one common world-Z correction so the **lowest source pickup placement** has center Z = **0.125 m**. Preserve the original height differences for all other variants.
- Frames **0–129**: center stays fixed; orientation is identity `wxyz = [1,0,0,0]`. Only the lowest pickup placement has its bottom face on `z=0`; higher placements remain above the ground.
- From frame **130**, including frame 130: retain the original wrist/contact rotation and use a **constant wrist-local translation**, calculated separately for each variant.
- Frame spacing is **0.033 s**, fps = 30.303030303; the pickup event is **4.29 s**. Robot motion, finger closure, timing and solved augmentation coordinates do not change.
- The previously requested **−0.05 m world-Z table shift** is also used on the website's cleaned point cloud and voxel mesh. Derive it from raw geometry once; do not shift an already lowered mesh.

Scope: these changes concern the under-table pickup object's 250 variants. They do not resize or reposition tabletop objects, bimanual boxes, rackets or ladder geometry.


## Frames and the exact selected-motion numbers

Updated correction: earlier notebook revisions grounded every variant separately. This revision preserves all source pickup-height differences with a single full-bank calibration. Variant 110 is about 7.857 mm above the ground; variant 20 is the grounded reference.

All lengths below are **metres**. World is right-handed, **+Z up**. Rotations map local vectors into world coordinates. Quaternions use **wxyz**, not xyzw.

The attached frame is **`left_rubber_hand`**, the source solver's contact frame. Its fixed URDF translation from `left_wrist_yaw_link` is `[0.1915, 0, 0]` m with zero RPY. The object offset below is relative to that contact frame, not directly to the wrist-yaw link. Do **not** change the URDF to implement this visualization edit. The earlier bimanual `0.1315` m right-hand change belongs to a different task.

The standalone selection is source variant **110**, at augmentation controls `[0.025, 0.044444444444444446, -0.10]` m (mean X, lower tie nearest mean Y, minimum Z).

| Quantity | Exact stored/calculated value |
|---|---|
| Original website hand-local object offset | `[-0.08, -0.01, 0.06]` |
| Requested world placement shift | `[0.0, -0.04, 0.0]` |
| Previous 20 cm package's center | `[0.5729764699935913, 0.26471149921417236, 0.10184981673955917]` |
| New stored initial center | `[0.5729764699935913, 0.22471149265766144, 0.13285687565803528]` |
| **New fixed contact-frame offset for variant 110** | **`[-0.0668109627349707, -0.05201963611860824, -0.07132226857250178]`** |
| Before-pickup quaternion | `[1, 0, 0, 0]` |
| Full-bank lowest original pickup center Z | `0.25760897994041443` (variant 20; controls `[0,0,-0.10]`) |
| Shared world-Z correction | `-0.13260897994041443` |
| Variant 110 initial bottom clearance | `0.007856875658035278` |
| Preserved pickup-height spread | `0.09631505608558655` |

The world shift and the local offset are different quantities. After pickup, the correction rotates with the hand. Adding `[0,-0.04,0]` to the carried object in every world frame would not preserve the requested rigid attachment.

Each variant has a different grasp pose, so **do not reuse variant 110's numeric local offset for every variant**. Apply the same world-placement rule, then solve its own local offset. Only the lowest source pickup placement is grounded. The others retain their original vertical separation, so the object height changes with the Z slider. Compute the reference once from the complete bank and pass it into subset exports; never recompute it from only the selected variant.


## Derivation

For variant $n$, let $h_{n,t}$ and $R_{n,t}$ denote the position and local-to-world rotation of `left_rubber_hand`. Let $k=130$ and $o_{old}=(-0.08,-0.01,0.06)$.

1. Recover the original visual placement from the immutable source:
   $c_{old,n}=h_{n,k}+R_{n,k}o_{old}$.
2. Compute the shared reference $z_{ref}=\min_n c_{old,n,z}$ and height correction $\Delta z=0.125-z_{ref}$. Set $c_n=(c_{old,n,x},\ c_{old,n,y}-0.04,\ c_{old,n,z}+\Delta z)$.
3. Solve $R_{n,k}o_n=c_n-h_{n,k}$ for the new local offset $o_n$.
4. For $t<k$, set object center to $c_n$ and rotation to identity.
5. For $t\ge k$, set center to $h_{n,t}+R_{n,t}o_n$ and rotation to $R_{n,t}$.

We use `np.linalg.solve` rather than a transpose because the saved float32 FK matrices have tiny orthogonality errors. For a mathematically exact rotation, both are equivalent. Original placement is cast to float32 first to reproduce the earlier website/RL export and its exact selected offset.

Position is continuous at pickup. Orientation switches to the original wrist rotation **at** frame 130; for variant 110 the switch is about **6.25°**. There is no orientation blend. A fractional-time renderer must keep identity for **every time < 4.29 s**; do not SLERP early from frame 129 into frame 130. The website samples frames with `floor((time + 1e-7) * fps)`, so it already follows this event boundary to numerical tolerance.


In [1]:
from pathlib import Path
import gzip, hashlib, json
import numpy as np
from scipy.spatial.transform import Rotation

# Works when launched from website/docs, the website root, the manuscript
# workspace, or the standalone package (and its subdirectories).
reference_original_z = None  # Full-bank calibration; explicit for subsets.
locations = [Path.cwd(), *Path.cwd().parents]
website = next((p for base in locations for p in [base, base/'website']
                if (p/'visualization/motions/tasks/under-table-pickup.npz').is_file()), None)
package = next((p for base in locations for p in [base, base/'exports/VICAR_Under_Table_Pick_RL']
                if (p/'reference/selected_source.npz').is_file()), None)
if website is not None:
    source_path = website/'visualization/motions/tasks/under-table-pickup.npz'
    with np.load(source_path, allow_pickle=False) as file:
        data = dict(file)
    data['metadata'] = json.loads(str(data['metadata']))
    variant_ids = np.arange(data['world_left_rubber_hand'].shape[0])
elif package is not None:
    source_path = package/'reference/selected_source.npz'
    with np.load(source_path, allow_pickle=False) as file:
        data = {key: file[key][None] for key in
                ['world_left_rubber_hand', 'rotation_left_rubber_hand']}
    package_meta = json.loads((package/'manifest.json').read_text())
    data['metadata'] = {'PICK_TIME': package_meta['object']['pick_frame'], 'fps': package_meta['fps']}
    variant_ids = np.array([package_meta['default_variant']])
    reference_original_z = package_meta['object']['ground_alignment']['reference_original_center_z_m']
else:
    raise FileNotFoundError('Run from the website or the standalone RL package directory.')
source_sha_before = hashlib.sha256(source_path.read_bytes()).hexdigest()
print(f'Loaded {len(variant_ids)} variant(s); {data["world_left_rubber_hand"].shape[1]} frames each.')
print('Source SHA256:', source_sha_before)


Loaded 250 variant(s); 385 frames each.
Source SHA256: 41936d8c2889379890f29abb6f0e1c56a9e5283b075a444f7eaf497cd5c4c384


## Reference implementation

The next cell is a copy of `visualization/under_table_object.py`, the helper used by the local Viser scene and the website exporter. This makes the notebook self-contained for the receiving agent. It returns float32 animation arrays and metadata with **every per-variant local offset**, indexed in exactly the source variant order. Re-running it derives from raw FK and the original visual offset, so the 4 cm shift does not accumulate.


In [2]:
"""Shared ground reference and rigid carrying for under-table pickup variants.

The source motion/URDF is read-only. See docs/UNDER_TABLE_OBJECT_PLACEMENT.ipynb
for the derivation, the exact variant-110 offset and executable handoff checks.
"""
import numpy as np
from scipy.spatial.transform import Rotation

OBJECT_DIMENSIONS = np.array([0.04, 0.04, 0.25])
SOURCE_LOCAL_OFFSET = np.array([-0.08, -0.01, 0.06])
WORLD_PLACEMENT_SHIFT = np.array([0.0, -0.04, 0.0])
TABLE_DISPLAY_SHIFT = np.array([0.0, 0.0, -0.05])


def under_table_object_motion(data, *, reference_original_z=None):
    """Return dimensions, [N,T,3] centers, [N,T,4] wxyz, and metadata.

    Apply the world shift once at placement, then solve a constant contact-frame
    offset separately for each variant. A single world-Z correction grounds only
    the lowest original pickup position and preserves all height differences.
    When processing a subset, pass reference_original_z from the full bank;
    never recalibrate the ground reference when switching the selected variant.
    The object is upright before PICK_TIME and uses the original wrist rotation
    from PICK_TIME onward. No blending,
    later floor clamping, robot changes or contact re-optimization is performed.
    """
    pick = int(data['metadata']['PICK_TIME'])
    hand = np.asarray(data['world_left_rubber_hand'], dtype=np.float64)
    rotation = np.asarray(data['rotation_left_rubber_hand'], dtype=np.float64)
    count, frames, _ = hand.shape
    if not 0 < pick < frames:
        raise ValueError('Pickup must be inside the motion.')

    # Reconstruct the ORIGINAL visual center, never an already shifted export.
    # Match the prior website/RL float32 placement before applying the new shift.
    original = (hand[:, pick] + rotation[:, pick] @ SOURCE_LOCAL_OFFSET).astype(np.float32).astype(np.float64)
    if reference_original_z is None:
        reference_original_z = float(original[:, 2].min())
    height_shift = OBJECT_DIMENSIONS[2] / 2 - reference_original_z
    placement = original + WORLD_PLACEMENT_SHIFT
    placement[:, 2] += height_shift
    # solve() also handles tiny non-orthogonality in stored float32 FK matrices.
    offsets = np.linalg.solve(rotation[:, pick], (placement - hand[:, pick])[..., None])[..., 0]
    positions = hand + np.einsum('ntij,nj->nti', rotation, offsets)
    positions[:, :pick] = placement[:, None]
    quaternions = Rotation.from_matrix(rotation.reshape(-1, 3, 3)).as_quat()[:, [3, 0, 1, 2]].reshape(count, frames, 4)
    quaternions[:, :pick] = [1, 0, 0, 0]
    presentation = dict(
        dimensions=OBJECT_DIMENSIONS.tolist(), groundZ=0, pickFrame=pick,
        pickTimeSeconds=pick / data['metadata']['fps'],
        attachedReferenceLink='left_rubber_hand',
        sourceLocalOffset=SOURCE_LOCAL_OFFSET.tolist(),
        worldPlacementShift=WORLD_PLACEMENT_SHIFT.tolist(),
        groundReferenceOriginalZ=reference_original_z, worldHeightShift=height_shift,
        groundingPolicy='One world-Z correction from the full-bank lowest pickup center; preserve inter-variant height differences',
        originalPlacement=original.tolist(), initialPosition=placement.tolist(),
        localOffsets=offsets.tolist(), tableDisplayShift=TABLE_DISPLAY_SHIFT.tolist(),
        beforePickup='Stationary; identity wxyz [1,0,0,0]; only the lowest pickup placement has its bottom face on z=0',
        fromPickup='Fixed per-variant contact-frame offset and unchanged wrist rotation',
        shiftScope='World -Y shift at placement only; offset rotates with the wrist after pickup',
        note='Display edit for all variants; robot/contact solve unchanged; later floor penetration is not clamped',
    )
    return OBJECT_DIMENSIONS.copy(), positions.astype(np.float32), quaternions.astype(np.float32), presentation


In [3]:
dimensions, centers, object_wxyz, presentation = under_table_object_motion(data, reference_original_z=reference_original_z)
pick = presentation['pickFrame']
selected = int(np.flatnonzero(variant_ids == 110)[0])
expected_offset = [-0.0668109627349707, -0.05201963611860824, -0.07132226857250178]
np.testing.assert_allclose(presentation['localOffsets'][selected], expected_offset, atol=1e-12, rtol=0)
np.testing.assert_array_equal(centers[selected, 0], np.array([0.5729764699935913, 0.22471149265766144, 0.13285687565803528], dtype=np.float32))
print('Dimensions (m):', dimensions.tolist())
print('Variant 110 initial center (m):', centers[selected, 0].tolist())
print('Variant 110 fixed left_rubber_hand offset (m):', presentation['localOffsets'][selected])
print('Grab event: frame', pick, 'at', presentation['pickTimeSeconds'], 'seconds')
print('Before pickup orientation:', object_wxyz[selected, pick-1].tolist())
print('At pickup orientation:', object_wxyz[selected, pick].tolist())


Dimensions (m): [0.04, 0.04, 0.25]
Variant 110 initial center (m): [0.5729764699935913, 0.22471149265766144, 0.13285687565803528]
Variant 110 fixed left_rubber_hand offset (m): [-0.0668109627349707, -0.05201963611860824, -0.07132226857250178]
Grab event: frame 130 at 4.29 seconds
Before pickup orientation: [1.0, 0.0, 0.0, 0.0]
At pickup orientation: [0.998512864112854, 0.0071300542913377285, 0.05307340249419212, 0.010220648720860481]


## Invariants across every variant

These checks verify placement, the shared ground reference and preserved vertical separation, all four bottom corners before pickup, unchanged carried orientation, position continuity and a constant local offset throughout carrying. They also verify that no source data was modified.


In [4]:
N, T, _ = centers.shape
np.testing.assert_array_equal(object_wxyz[:, :pick], np.broadcast_to([1,0,0,0], (N,pick,4)))
np.testing.assert_array_equal(centers[:, :pick], np.broadcast_to(centers[:, :1], (N,pick,3)))
original = np.asarray(presentation['originalPlacement'])
np.testing.assert_allclose(centers[:,0,:2]-original[:,:2], np.tile([0,-.04], (N,1)), atol=3e-8, rtol=0)
base_corners = np.array([[-.02,-.02,-.125],[-.02,.02,-.125],[.02,-.02,-.125],[.02,.02,-.125]])
clearance = original[:,2] - presentation['groundReferenceOriginalZ']
np.testing.assert_allclose(centers[:,:pick,None,2] + base_corners[None,None,:,2], np.broadcast_to(clearance[:,None,None],(N,pick,4)), atol=1e-8, rtol=0)
assert clearance.min() >= -1e-8
np.testing.assert_allclose(centers[:,0,2]-centers[0,0,2], original[:,2]-original[0,2], atol=1e-8, rtol=0)
if website is not None:
    assert abs(clearance.min()) < 1e-8
    assert np.count_nonzero(clearance < 1e-8) == 1
    assert clearance.max() > 0.09
np.testing.assert_allclose(centers[:,pick], centers[:,pick-1], atol=1e-7, rtol=0)
hand = data['world_left_rubber_hand'].astype(float)
R = data['rotation_left_rubber_hand'].astype(float)
recovered_offsets = np.linalg.solve(R[:,pick:], (centers[:,pick:]-hand[:,pick:])[...,None])[...,0]
np.testing.assert_allclose(recovered_offsets, np.broadcast_to(recovered_offsets[:,:1], recovered_offsets.shape), atol=1e-7, rtol=0)
expected_rotations = Rotation.from_matrix(R[:,pick:].reshape(-1,3,3))
actual_rotations = Rotation.from_quat(object_wxyz[:,pick:].reshape(-1,4)[:,[1,2,3,0]])
assert np.max((actual_rotations.inv()*expected_rotations).magnitude()) < 2e-7
assert hashlib.sha256(source_path.read_bytes()).hexdigest() == source_sha_before
print(f'PASS: all {N} variants upright at their original relative heights before pickup, fixed wrist attachment afterward, source file unchanged.')


PASS: all 250 variants upright at their original relative heights before pickup, fixed wrist attachment afterward, source file unchanged.


## Check the actual published animation pack and standalone export

The website reads compressed float32 channels in `[variant, frame, channel]` order. Its base Viser scene provides the box geometry; the dynamic pack supplies center and quaternion per frame. Both must be regenerated. This check compares the files the browser actually loads to the function above. The optional standalone comparison allows a few floating-point quaternion bits to differ due to the RL export's normalization; physical rotations remain the same.


In [5]:
if website is not None:
    public = website/'assets/augmentation/tasks'
    config = json.loads((public/'under-table-pickup.json').read_text())
    spec = config['dynamic']
    pack = np.frombuffer(gzip.decompress((public/spec['file']).read_bytes()), dtype='<f4').reshape(N,T,spec['frameStride'])
    for prop, expected in [('position',centers), ('wxyz',object_wxyz)]:
        channel = next(c for c in spec['channels'] if c['node']=='/object_cuboid' and c['property']==prop)
        actual = pack[:,:,channel['offset']:channel['offset']+channel['width']]
        np.testing.assert_array_equal(actual, expected)
    np.testing.assert_allclose(config['objectPresentation']['localOffsets'], presentation['localOffsets'], atol=1e-12, rtol=0)
    print('PASS: website binary channels and JSON offsets match all variants.')
if package is not None:
    with np.load(package/'motions/default.npz', allow_pickle=False) as motion:
        np.testing.assert_array_equal(centers[selected], motion['object_pos_w'])
        np.testing.assert_allclose(object_wxyz[selected], motion['object_quat_w'], atol=2e-7, rtol=0)
    print('PASS: variant 110 matches the standalone RL package.')


PASS: website binary channels and JSON offsets match all variants.
PASS: variant 110 matches the standalone RL package.


## Website integration and regeneration

The implementation is already connected at these repository paths:

1. `visualization/under_table_object.py`: pure placement/attachment calculation above.
2. `visualization/export_tasks.py`: `TaskScene._manipulation()` uses the helper only for `under-table-pickup`; `table_geometry()` cleans the extra leg, then lowers both display layers by 5 cm. The source collision proxy and contact-diagnostic calculations retain the original solver geometry.
3. `assets/augmentation/tasks/under-table-pickup-base.viser`: updated 25 cm box and lowered table; fallback recording uses the same object poses as the dynamic channels.
4. `assets/augmentation/tasks/under-table-pickup.json`: `objectPresentation.localOffsets`, `initialPosition` and `originalPlacement` contain one entry per variant, matching `shifts` order.
5. `assets/augmentation/tasks/under-table-pickup-objects-<hash>.bin.gz`: updated position/wxyz channels. A content hash prevents an old cached object pack from being paired with the new geometry.
6. `assets/viewer-manifest.js` and its URL in `index.html`: regenerated/versioned so the browser loads the updated scene and metadata. Joint-rotation data and source motion NPZ stay unchanged.
7. `viser-client-extension/AugmentationPlayback.ts`: existing `apply()` chooses the current integer frame and sends absolute `position`/`wxyz` values. No frontend transform should add a second shift, tilt, or table translation.

From the website repository root:

```sh
python -m visualization export --task under-table-pickup
python -m unittest visualization.test_tasks -v
npm test
python -m http.server 8000
```

The export command also writes the manifest. Inspect before and after frame 130, then try all XYZ extremes and variant 110. The default website selection stays variant 112; standalone variant 110 is selected by X=0.025, Y=0.044444444444444446, Z=-0.10.

Commit the helper, exporter, tests, notebook, regenerated base scene, JSON, hashed dynamic pack, manifest and cache key on branch `website`. GitHub Pages deploys that branch's root. All 250 variants remain available; other task scenes remain unchanged.


## RL and interpretation notes

For an RL export, save the new object dimensions/center/orientation and recompute object linear/angular velocities; update both visual and collision geometry and inertia. The standalone package already does this and retains only variant 110. Its manifest carries the full-bank ground calibration; do not re-ground that subset separately. Its robot arrays are unchanged. Do not overwrite robot velocities with object derivatives.

Ground alignment calibrates the **lowest pre-pickup placement across the full variant bank**. It does not flatten all variants to the floor, and it is not a clamp applied across the temporal path. Once attached, the unchanged wrist motion can take part of the cuboid below ground. The selected motion reaches about **−0.11482 m** at its lowest object point. This notebook deliberately does not clamp Z after pickup, because that would break the fixed wrist attachment. The source contact solve has not been rerun for the new object geometry or lowered visual table. These are kinematic visualization/reference changes, not a validated physics rollout.

For another renderer, keep the initial transform for `time < pick_time`; from the event onward use the wrist transform with the per-variant offset. Recompute offsets only when the source grasp transform or requested placement changes. Never repeatedly add the 4 cm shift to the last rendered object position.


In [6]:
world_rotation = Rotation.from_quat(object_wxyz[selected][:,[1,2,3,0]]).as_matrix()
lowest_z = centers[selected,:,2] - np.abs(world_rotation[:,2,:]) @ (dimensions/2)
print('Shared original reference Z (m):', presentation['groundReferenceOriginalZ'])
print('Shared world Z correction (m):', presentation['worldHeightShift'])
print('Variant 110 bottom Z before pickup (m):', float(lowest_z[:pick].min()))
print('Variant 110 lowest object point over full clip (m):', float(lowest_z.min()))
print('Grounding after pickup is not imposed; the fixed wrist attachment is preserved.')


Shared original reference Z (m): 0.25760897994041443
Shared world Z correction (m): -0.13260897994041443
Variant 110 bottom Z before pickup (m): 0.007856875658035278
Variant 110 lowest object point over full clip (m): -0.11481656669651952
Grounding after pickup is not imposed; the fixed wrist attachment is preserved.


## Robot embodiment update — 14 October 2026

Under-table pickup now uses the fixed Unitree rubber hand on the right wrist and retains the left Dex3. The website URDF is `visualization/robot-hands/g1_under_table.urdf`; regenerate it with `python scripts/build_under_table_robot.py`. The right hand CAD link, `right_unitree_hand`, is mounted at `[0.0415,-0.003,0]` m from `right_wrist_yaw_link`, with zero RPY. This is separate from the solver contact frame `right_rubber_hand`, which remains at `[0.1915,0,0]` in the standalone package.

The website maps its unchanged 43-column source data to the 36 retained joints by name. The standalone RL package exports 36 joint columns and 49 body frames; the seven right-finger DOFs and their links are removed. Its replacement hand includes the supplied 0.170 kg inertial and matching mesh collision. Retained joint trajectories and common body states are unchanged. The left-hand object transforms and shared height calibration described above are unchanged.
